# H2I Lab OPERA Workflow — Local Walkthrough

This notebook exercises the extracted `subside_analysis.h2i_lab` package end-to-end against real Earthdata downloads — the same code path the Tapis batch app runs via `run.sh`.

Stages:
1. **Setup** — path, output dir, Earthdata credentials
2. **Config** — build a small `H2IRunConfig` over a tiny AOI
3. **Preflight** — frame discovery + product search
4. **Run** — download/subset NetCDFs, build the Folium preview, zip archive
5. **Inspect** — show the manifest + the preview overlay

> Requires the `subside-h2i-opera` conda environment (see `environment.yaml`). A real run will download a few GB of NetCDFs over Earthdata.

## 1. Setup

Put the repo's `subside/` directory on `sys.path` so `subside_analysis` is importable from this notebook's location (`subside/workflow_apps/h2i_lab/`).

In [ ]:
import json, os, sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd()).resolve()
SUBSIDE_ROOT = NOTEBOOK_DIR.parents[1]  # subside/
sys.path.insert(0, str(SUBSIDE_ROOT))

OUTPUT_DIR = NOTEBOOK_DIR / "walkthrough_outputs"
AOI_GEOJSON = NOTEBOOK_DIR / "walkthrough_aoi.geojson"
OUTPUT_DIR.mkdir(exist_ok=True)
print("subside/ on path:", SUBSIDE_ROOT)
print("outputs:", OUTPUT_DIR)

## 2. Earthdata credentials

The runner accepts `EARTHDATA_USERNAME` + `EARTHDATA_PASSWORD` env vars, or a standard `~/.netrc` entry for `urs.earthdata.nasa.gov`. The cell below just confirms which path you're on — set the variables (or write the netrc entry) before running the next step.

In [ ]:
from netrc import netrc

if os.environ.get("EARTHDATA_USERNAME") and os.environ.get("EARTHDATA_PASSWORD"):
    print("Using EARTHDATA_USERNAME env credentials.")
else:
    try:
        creds = netrc().authenticators("urs.earthdata.nasa.gov")
        print("Using ~/.netrc credentials." if creds else "No urs.earthdata.nasa.gov entry in ~/.netrc.")
    except FileNotFoundError:
        print("No env credentials and no ~/.netrc. Set creds before running the next cell.")

## 3. Run config + AOI

Tiny AOI over the Houston-Galveston area (known subsidence + good DISP-S1 coverage). Adjust the bounding box and date range freely; smaller windows download faster.

In [ ]:
from subside_analysis.h2i_lab.config import H2IRunConfig

aoi = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {},
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [-95.55, 29.55], [-95.35, 29.55],
                [-95.35, 29.75], [-95.55, 29.75],
                [-95.55, 29.55],
            ]],
        },
    }],
}
AOI_GEOJSON.write_text(json.dumps(aoi))

config = H2IRunConfig.from_dict({
    "start_date": "2024-06-01",
    "end_date": "2024-09-01",
    "output_dir": str(OUTPUT_DIR),
    "aoi_geojson_path": str(AOI_GEOJSON),
    "num_workers": 2,
    "min_overlap_percent": 50.0,
})
config

## 4. Preflight

Downloads the OPERA frames index, finds frames intersecting the AOI, queries ASF for available DISP-S1 products, filters by date, and writes `outputs/preflight-manifest.json`. No Earthdata download yet — this is the cheap discovery step.

In [ ]:
from subside_analysis.h2i_lab.runner import preflight

preflight_manifest = preflight(config)
print("Frames :", preflight_manifest["frame_ids"])
print("Products:", preflight_manifest["product_count"])
print("Warnings:", preflight_manifest["warnings"])

## 5. Run — download, subset, preview, archive

Runs the full pipeline: pixel-bbox estimate, parallel NetCDF download with cropping, Folium PNG/HTML preview, and a zip archive of the cropped products. Output paths land under `outputs/`.

In [ ]:
from subside_analysis.h2i_lab.runner import run

run_manifest = run(config)
run_manifest["artifacts"]

## 6. Inspect artifacts

Render the overlay PNG and the Folium preview HTML produced by the run.

In [ ]:
from IPython.display import IFrame, Image, display

artifacts = run_manifest["artifacts"]
overlay = artifacts.get("overlay_png")
preview = artifacts.get("preview_html")
if overlay:
    display(Image(filename=overlay))
if preview:
    display(IFrame(src=preview, width="100%", height=420))

print("NetCDFs downloaded:", len(artifacts.get("downloaded_files", [])))
print("Archive zip      :", artifacts.get("archive_zip"))